#have to define configuration in pipeline:: orders, inventory, source_directory

#IMPORTS + CONFIG LOAD

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

try:
    source_directory = spark.conf.get("source_directory")
except Exception as e:
    raise ValueError("Missing 'source_directory' in DLT Pipeline Configuration!")

#Schema definition

Transactions Schema

In [0]:
transactions_schema = StructType([
    StructField("transaction_date", StringType()),
    StructField("stock_date", StringType()),
    StructField("product_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("quantity", IntegerType())
])

Stores Schema

In [0]:
stores_schema = StructType([
    StructField("store_id", IntegerType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("store_type", StringType(), True),
    StructField("store_name", StringType(), True),
    StructField("store_street_address", StringType(), True),
    StructField("store_city", StringType(), True),
    StructField("store_state", StringType(), True),
    StructField("store_country", StringType(), True),
    StructField("store_phone", StringType(), True),
    StructField("first_opened_date", StringType(), True),   # parse later
    StructField("last_remodel_date", StringType(), True), # parse later
    StructField("total_sqft", IntegerType(), True),
    StructField("grocery_sqft", IntegerType(), True)    
])

Regions Schema

In [0]:
regions_schema = StructType([
    StructField("region_id", IntegerType()),
    StructField("sales_district", StringType()),
    StructField("sales_region", StringType())
])

Returns Schema

In [0]:
returns_schema = StructType([
    StructField("return_date", StringType()),
    StructField("product_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("quantity", IntegerType())
])

Calendar Schema

In [0]:
calendar_schema = StructType([
    StructField("date", StringType())
])

##Ingesting Transactions Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_raw_transactions")
def bronze_transactions():

    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")                     
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .schema(transactions_schema)
        .load(f"{source_directory}/transactions/")
        .withColumn("ingestion_timestamp", current_timestamp())  
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("_source_system", lit("adls"))  
    )

##Ingesting Stores Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_raw_stores")
def bronze_stores():

    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("pathGlobFilter", "stores") 
        .schema(stores_schema)
        .load(source_directory)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("_source_system", lit("adls"))
    )

##Ingesting Regions Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_raw_regions")
def bronze_regions():

    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("pathGlobFilter", "regions")  
        .schema(regions_schema)
        .load(source_directory)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("_source_system", lit("adls"))
    )

##Ingesting Returns Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_raw_returns")
def bronze_returns():

    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("pathGlobFilter", "returns") 
        .schema(returns_schema)
        .load(source_directory)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("_source_system", lit("adls"))
    )

##Ingesting Calendar Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_raw_calendar")
def bronze_calendar():

    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("pathGlobFilter", "calendar") 
        .schema(calendar_schema)
        .load(source_directory)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("_source_system", lit("adls"))
    )

In [0]:
#from pyspark.sql.functions import *

def read_kafka_stream(topic_name):
    """
    Reads a stream from Kafka safely inside DLT.
    Fetch secrets INSIDE function (DLT-safe)
    """

    
    kafka_server = dbutils.secrets.get(scope = "kafka-connector", key = "kafka-bootstrap")
    kafka_api_key = dbutils.secrets.get(scope = "kafka-connector", key = "kafka-api-key")
    kafka_secret = dbutils.secrets.get(scope = "kafka-connector", key = "kafka-api-secret")

    # Kafka config
    kafka_options = {
        "kafka.bootstrap.servers": kafka_server,
        "subscribe": topic_name,
        "kafka.security.protocol": "SASL_SSL",
        "kafka.sasl.mechanism": "PLAIN",
        "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{kafka_api_key}" password="{kafka_secret}";',
        "startingOffsets": "earliest"   # ensures no data loss
    }

    return (
        spark.readStream
        .format("kafka")
        .options(**kafka_options)
    )

##Ingesting Inventory Data

In [0]:
inventory_topic = spark.conf.get("inventory")

@dlt.table(name="maven_market_uc.bronze.bronze_raw_inventory")
def bronze_kafka_inventory():

    return (
        read_kafka_stream(inventory_topic).load()   # helper
        .selectExpr(
            "CAST(value AS STRING) as raw_payload",
            "topic as _kafka_topic",
            "partition as _kafka_partition",
            "offset as _kafka_offset",
            "timestamp as _kafka_timestamp"
        )
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit("kafka"))
    )

##Ingesting Orders Data

In [0]:
orders_topic = spark.conf.get("orders")

@dlt.table(name="maven_market_uc.bronze.bronze_raw_orders")
def bronze_kafka_orders():

    return (
        read_kafka_stream(orders_topic).load()   # helper
        .selectExpr(
            "CAST(value AS STRING) as raw_payload",
            "topic as _kafka_topic",
            "partition as _kafka_partition",
            "offset as _kafka_offset",
            "timestamp as _kafka_timestamp"
        )
        .withColumn("ingestion_timestamp", current_timestamp())  
        .withColumn("_source_system", lit("kafka"))
    )

##Customers Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_customers")
def bronze_customers():

    return (
        spark.read.table("maven_market_uc.bronze.bronze_raw_customers")
        .withColumn("_source_system", lit("fivetran"))
    )

##Products Data

In [0]:
@dlt.table(name="maven_market_uc.bronze.bronze_products")
def bronze_products():

    return (
        spark.read.table("maven_market_uc.bronze.bronze_raw_products")
        .withColumn("_source_system", lit("fivetran"))
    )